In [1]:
import os

print(os.listdir("../data"))

['student_dataset.csv']


In [41]:
import pandas as pd
df = pd.read_csv("../data/student_dataset.csv")
df.head()

,StudentID,StudyHours,Attendance,PreviousScore,AssignmentScore,SleepHours,PreviousFailures,Result
0,S001,2.0,62,45,48,5.0,2,Fail
1,S002,3.0,68,52,55,6.0,1,Fail
2,S003,4.0,75,58,62,6.5,1,Pass
3,S004,5.0,82,65,70,7.0,0,Pass
4,S005,1.5,55,40,42,5.0,3,Fail


In [4]:
%pip install pandas numpy scikit-learn joblib matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import sklearn
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("All packages imported successfully!")
print("Joblib version:", joblib.__version__)
print("Scikit-learn version:", sklearn.__version__)


All packages imported successfully!
Joblib version: 1.4.2
Scikit-learn version: 1.8.0


In [3]:
#Load the Dataset
data_path = "../data/student_dataset.csv"
df = pd.read_csv(data_path)
print("Dataset loaded successfully!")


Dataset loaded successfully!


In [4]:
df.head()

,StudentID,StudyHours,Attendance,PreviousScore,AssignmentScore,SleepHours,PreviousFailures,Result
0,S001,2.0,62,45,48,5.0,2,Fail
1,S002,3.0,68,52,55,6.0,1,Fail
2,S003,4.0,75,58,62,6.5,1,Pass
3,S004,5.0,82,65,70,7.0,0,Pass
4,S005,1.5,55,40,42,5.0,3,Fail


In [5]:
df.shape

(50, 8)

In [6]:
df.columns

Index(['StudentID', 'StudyHours', 'Attendance', 'PreviousScore',
       'AssignmentScore', 'SleepHours', 'PreviousFailures', 'Result'],
      dtype='str')

In [7]:
df.info

<bound method DataFrame.info of    StudentID  StudyHours  Attendance  PreviousScore  AssignmentScore  \
0       S001         2.0          62             45               48   
1       S002         3.0          68             52               55   
2       S003         4.0          75             58               62   
3       S004         5.0          82             65               70   
4       S005         1.5          55             40               42   
5       S006         6.0          88             72               78   
6       S007         2.5          64             48               50   
7       S008         7.0          92             80               85   
8       S009         3.5          70             55               60   
9       S010         2.0          58             43               45   
10      S011         5.5          85             68               75   
11      S012         4.5          78             61               65   
12      S013         1.0        

In [39]:
#check for missing values and duplicate records
df.duplicated().sum()

np.int64(0)

In [38]:
#Understand the Target Variable
df["Result"].unique()

<ArrowStringArray>
['Fail', 'Pass']
Length: 2, dtype: str

In [20]:
#Preapare feature columns that are required for prediction, ignore unwanted columns(student_id)from the list
features = [
    "StudyHours",
    "Attendance",
    "PreviousScore",
    "AssignmentScore",
    "SleepHours",
    "PreviousFailures",
]
target = "Result"
X = df[features]  #  return  only given columns values 
Y = df[target]     # return only churn column values 
print( X.head() )
print( Y.head() )


   StudyHours  Attendance  PreviousScore  AssignmentScore  SleepHours  \
0         2.0          62             45               48         5.0   
1         3.0          68             52               55         6.0   
2         4.0          75             58               62         6.5   
3         5.0          82             65               70         7.0   
4         1.5          55             40               42         5.0   

   PreviousFailures  
0                 2  
1                 1  
2                 1  
3                 0  
4                 3  
0    Fail
1    Fail
2    Pass
3    Pass
4    Fail
Name: Result, dtype: str


In [21]:
#convert pass/fail into numerical values
Y = Y.map({
    "Fail": 0,
    "Pass": 1
})
display(Y.head())

0    0
1    0
2    1
3    1
4    0
Name: Result, dtype: int64

In [22]:
#split the dataset into training and testing data
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    stratify=Y
)
print("Total records :", len(X))
print("Training data :", len(X_train))
print("Testing data  :", len(X_test))

Total records : 50
Training data : 40
Testing data  : 10


In [23]:
#perform feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
# Create the ML Model
model = LogisticRegression(
    random_state=42
)

In [26]:
# train the model
model.fit(
    X_train_scaled,
    Y_train
)
print("Model training completed!")

Model training completed!


In [40]:
# Make Predictions on test data
Y_pred = model.predict(
    X_test_scaled
)
print("Predictions:")
print(Y_pred)

Predictions:
[0 1 0 0 1 1 1 1 0 1]


In [29]:
# compare actual vs predicted
comparison = pd.DataFrame({
    "Actual": Y_test.values,
    "Predicted": Y_pred
})
comparison

,Actual,Predicted
0,0,0
1,1,1
2,0,0
3,0,0
4,1,1
5,1,1
6,1,1
7,1,1
8,0,0
9,1,1


In [30]:
import joblib
#Save the Trained Model
model_path = "../models/Result_model.pkl"
joblib.dump(
    model,
    model_path
)
print("Model saved successfully!")

Model saved successfully!


In [31]:
# save the scalar
scaler_path = "../models/scaler.pkl"
joblib.dump(
    scaler,
    scaler_path
)
print("Scaler saved successfully!")


Scaler saved successfully!


In [33]:
# load  the model 
loaded_model = joblib.load(
    "../models/Result_model.pkl"
)
loaded_scaler = joblib.load(
    "../models/scaler.pkl"
)
print("Model and scaler loaded successfully!")

Model and scaler loaded successfully!


In [34]:
#New Customer Prediction
# load the new customer
new_customer = pd.DataFrame([
    {
         "StudyHours": 6,
         "Attendance": 85,
         "PreviousScore": 72,
         "AssignmentScore": 80,
         "SleepHours": 7,
         "PreviousFailures": 0
    }
])
new_customer

,StudyHours,Attendance,PreviousScore,AssignmentScore,SleepHours,PreviousFailures
0,6,85,72,80,7,0


In [36]:
# The model was trained using scaled data. 
# So we must scale the new customer's data using the same saved scaler.
new_customer_scaled = loaded_scaler.transform(
    new_customer
)
#prediction
prediction = loaded_model.predict(
    new_customer_scaled
)
print("Predictions:", prediction)

Predictions: [1]


In [37]:
#prediction 
if prediction[0] == 1:
    print("Result: Pass")
else:
    print("Result: Fail")

Result: Pass
